## OpsinMap3D

Opsin homology using structures. Maps residues in each query to the reference using structure alignment.

In [ ]:
#@title Inputs

#@markdown 1. Open the `🗀 Files` panel to the left
#@markdown 2. Upload your query PDB file(s) in the root directory
#@markdown 3. Select the family
FAMILY = "canonical microbial rhodopsins" #@param [ "canonical microbial rhodopsins", "heliorhodopsins", "animal rhodopsins" ]
#@markdown 4. Choose alignment methods (**mtm_align_msa** is recommended)
mtm_align_msa = True # @param {"type":"boolean"}
mustang_msa = False # @param {"type":"boolean"}
sap_pair = False # @param {"type":"boolean"}
t_coffee_msa = False # @param {"type":"boolean"}
mafft_msa = False # @param {"type":"boolean"}
probcons_msa = False # @param {"type":"boolean"}
#@markdown 4. Adjust other settings if needed
TEMPLATES = "only experimental" #@param [ "only experimental", "prefer experimental", "any" ]
#@markdown &nbsp;&nbsp;&nbsp;&nbsp;&nbsp; ↳ _template selection: **only experimental** structures, **prefer experimental**, or recruite **any** template_
MAX_PCT = 100 # @param {"type":"integer"}
N = 20 # @param {"type":"integer"}
#@markdown &nbsp;&nbsp;&nbsp;&nbsp;&nbsp; ↳ _maximum sequence identity and maxmimum number of templates_
PAD_N = 30 # @param {"type":"integer"}
PAD_C = 30 # @param {"type":"integer"}
#@markdown &nbsp;&nbsp;&nbsp;&nbsp;&nbsp; ↳ _Padding (residues added to trimmed domain sequence)_

#@markdown 5. Press `▹ Run all` to run the workflow

# Check the inputs and the arguments

from pathlib import Path

MAX_SEQ_ID = MAX_PCT / 100

PREFER_EXPTL = TEMPLATES == "prefer experimental"
ONLY_EXPTL = TEMPLATES == "only experimental"

methods = []
if mtm_align_msa:
    methods.append('mtm_align_msa')
if mustang_msa:
    methods.append('mustang_msa')
if sap_pair:
    methods.append('sap_pair')
if probcons_msa:
    methods.append('probcons_msa')
if t_coffee_msa:
    methods.append('t_coffee_msa')
if mafft_msa:
    methods.append('mtm_align_msa')
METHODS = ','.join(methods)

assert METHODS, "No methods selected!"

if FAMILY == "heliorhodopsins":
    DATASET = "heliorhodopsin"
elif FAMILY == "animal rhodopsins":
    DATASET = "animal_rhodopsin"
else:
    DATASET = "canon_microb_rhodopsin"

PDB_FILES = list(Path('.').glob('*.pdb', case_sensitive=False))

if not PDB_FILES:
    raise FileNotFoundError(f"No PDB files found in the folder and no accession provided")

# Environment variables

import os

ROOT = "/content/mmb"

os.environ['MAMBA_ROOT_PREFIX'] = ROOT
os.environ['PATH'] = f":{ROOT}/bin:{ROOT}/condabin:/content/bin/:" + os.environ['PATH']
os.environ['CONDA_PREFIX'] = ROOT
os.environ['CONDA_SHLVL'] = '1'
os.environ['CONDA_DEFAULT_ENV'] = 'base'
os.environ['CONDA_PROMPT_MODIFIER'] = 'base'

In [ ]:
#@title Setup the environment

%%bash

ENV_DIR=/content/mmb/envs/env

if [ ! -s /usr/bin/micromamba ]; then
    echo "Downloading micromamba"
    url=https://micro.mamba.pm/api/micromamba/linux-64/latest
    wget -qO- $url | tar -C /usr/ -xj bin/micromamba &
fi
if [ -z "$(command -v zstd)" ]; then
    echo "Installing zstd"
    apt install -y zstd &> /dev/null
fi
wait
if [ ! -d $ENV_DIR ]; then
    echo "Setting up the environmemt"
    mkdir -p $ENV_DIR
    url=https://github.com/BejaLab/opsintools/releases/download/v0.5-beta/opsintools-v0.5-beta_conda_env_linux-64.tar.zst
    wget -qO- $url | tar -I "zstd -T4" -xf - -C $ENV_DIR
    (cd $ENV_DIR && ./bin/conda-unpack)
fi
if [ ! -d data ]; then
    micromamba run -n env opsindata -d data
fi

In [ ]:
#@title Run the pipeline

from google.colab import files
import sys

DATA_PATH = next(Path('data').glob('v*'))

flags = []

if PREFER_EXPTL:
  flags.append('--prefer-exptl')
if ONLY_EXPTL:
  flags.append('--only-exptl')

OUTPUT_PATH = Path('output')
OUTPUT_PATH.mkdir(exist_ok = True)
dataset_dir = DATA_PATH / DATASET

flags_str = ' '.join(flags)

for pdb_path in PDB_FILES:
  this_output_path = OUTPUT_PATH / pdb_path.name
  !micromamba run -n env opsinmap3d -f -d {dataset_dir} -i {pdb_path} -o {this_output_path} --methods {METHODS} --pad-n {PAD_N} --pad-c {PAD_C} --max-seq-id {MAX_SEQ_ID} {flags_str}
  if _exit_code:
      files.download(OUTPUT_PATH / 't_coffee.aln.log')
      sys.exit(1)


In [ ]:
#@title Output

colab_utils_path = Path("colab_utils.py")
if not colab_utils_path.exists():
  import urllib.request
  urllib.request.urlretrieve(url = "https://raw.githubusercontent.com/BejaLab/opsintools/refs/heads/main/colab/colab_utils.ipynb", filename = colab_utils_path)

from google.colab import files
import shutil
import colab_utils

colab_utils.generate_html_report(OUTPUT_PATH, DATA_PATH, def_profile = DATASET)
shutil.make_archive('opsinmap3d_output', 'zip', 'output')
files.download("opsinmap3d_output.zip")